In [2]:
import os
from dotenv import load_dotenv, find_dotenv

env_pfad = find_dotenv()          # sucht die Datei .env vom Notebook-Ordner aufwärts
load_dotenv(env_pfad)

print("Datei .env gefunden:", env_pfad)
print("API key geladen:", os.getenv("ANTHROPIC_API_KEY") is not None)

Datei .env gefunden: /Users/alpdalkilic/Development/ki-lernjournal/.env
API key geladen: True


In [3]:
from pathlib import Path

pfad = Path("data/ki_verordnung_auszug_2024.txt")
text = pfad.read_text(encoding="utf-8")

print(len(text), "Zeichen")
print(text[:400])

38086 Zeichen
Artikel 5 | Verbotene Praktiken im KI-Bereich
(1) Folgende Praktiken im KI-Bereich sind verboten: a) das Inverkehrbringen, die Inbetriebnahme oder die Verwendung eines KI-Systems, das Techniken der unterschwelligen Beeinflussung außerhalb des Bewusstseins einer Person oder absichtlich manipulative oder täuschende Techniken mit dem Ziel oder der Wirkung einsetzt, das Verhalten einer Person oder ein


## Zerlegung: chunking mit source

1. Eine leere Liste für das Ergebnis anlegen. Dazu einen Merker „Artikel“, der am Anfang leer ist.
2. Den Text in einzelne Zeilen zerlegen. Ergebnis: eine Liste mit 62 Zeilen (6 Kopfzeilen, 50 Absatzzeilen, 6 Leerzeilen).
3. Zeile für Zeile durchgehen und je Zeile den Typ bestimmen:
   * Beginnt die Zeile mit „Artikel“, dann die Nummer zwischen „Artikel “ und „ |“ herauslösen und im Merker „Artikel“ ablegen. Es entsteht kein Eintrag in der Liste. Der Merker gilt ab jetzt für alle folgenden Zeilen, bis die nächste Artikel-Zeile ihn überschreibt.
   * Beginnt die Zeile mit „(“, dann die Zahl zwischen „(“ und „)“ herauslösen. Das ist die Absatznummer. Aus dem Merker „Artikel“ und der Absatznummer die source zusammensetzen: „Art. 26 Abs. 6“. Ein dictionary mit der ganzen Zeile als text und der source anlegen und an die Ergebnisliste anhängen.
   * Ist die Zeile leer, dann nichts tun und zur nächsten Zeile gehen.
4. Am Ende prüfen: Die Liste hat 50 Einträge, der erste hat die source „Art. 5 Abs. 1“, der letzte „Art. 50 Abs. 7“, und der chunk „Art. 26 Abs. 6“ enthält „sechs Monaten“.

In [13]:
chunks = []
artikel = None

for zeile in text.splitlines():
    zeile = zeile.strip()

    if zeile.startswith('Artikel'):
        artikel = zeile.split(' | ')[0].replace('Artikel ', '')   # "Artikel 26 | Pflichten …" → "Artikel 26" → "26"


    elif zeile.startswith('('):
        absatz = zeile.split(')')[0].replace('(', '')             # "(6) Betreiber …" → "(6" → "6"

        chunks.append({
            'source': f'Art. {artikel} Abs. {absatz}',
            'text': zeile
        })

In [ ]:
print(len(chunks))                 # erwartet: 50
print(chunks[0]['source'])         # erwartet: Art. 5 Abs. 1
print(chunks[-1]['source'])        # erwartet: Art. 50 Abs. 7

for chunk in chunks:
    if chunk['source'] == 'Art. 26 Abs. 6':
        print('sechs Monaten' in chunk['text'])   # erwartet: True


In [16]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 1. Modell laden (liegt seit dem 24.11. auf dem Air, wird nicht neu heruntergeladen)
model = SentenceTransformer("intfloat/multilingual-e5-small")

# 2. prefix vor jeden Text setzen: aus "(1) Folgende …" wird "passage: (1) Folgende …"
texte = ["passage: " + chunk["text"] for chunk in chunks]

# 3. embedding rechnen; normalize_embeddings=True bringt jeden vector auf die norm 1
vectors = model.encode(texte, normalize_embeddings=True)

print(type(vectors))
print(vectors.shape)
print(np.linalg.norm(vectors[0]))   # norm des ersten vectors


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8395.95it/s]


<class 'numpy.ndarray'>
(50, 384)
1.0
